# GFCM quickstart

A tail-sensitive, mixed-type conditional-independence test: it tests **H0: X independent of Y given Z**, and unlike partial correlation / GCM it detects dependence in the *scale* and *tails*, not just the conditional covariance.

This notebook covers: a basic test, configuration, mixed-type data, and use inside constraint-based discovery (PC).

In [ ]:
import os, sys
# make the repo importable when running from examples/ (skip if gfcm is pip-installed)
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'experiments'))

import numpy as np
from gfcm import unified_test, GFCMConfig, infer_types
print('gfcm ready')

## 1. A basic conditional-independence test

`unified_test(X, Y, Z)` returns a p-value. Small p (< 0.05) rejects independence. We build a **null** (Y depends on Z only) and a **scale alternative** (X drives Y's spread while the conditional mean and covariance stay flat -- the kind of edge a covariance test misses).

In [ ]:
rng = np.random.default_rng(0)
n = 2000
Z = rng.normal(size=(n, 2))
X = Z[:, 0] + rng.normal(size=n)

Y_null = Z[:, 1] + rng.normal(size=n)                                   # X _||_ Y | Z
Y_alt  = Z[:, 1] + np.exp(0.6 * (X - Z[:, 0])) * rng.standard_t(4, n)   # X drives Y's scale

print('null p =', round(unified_test(X, Y_null, Z), 4), ' (expect large)')
print('alt  p =', round(unified_test(X, Y_alt,  Z), 4), ' (expect ~0)')

## 2. Configuration

Configuration is explicit via the frozen `GFCMConfig` dataclass -- **no environment variables**. The defaults are the canonical published configuration, so `unified_test(X, Y, Z)` reproduces the paper's test. Construct a modified `GFCMConfig(...)` for an ablation.

In [ ]:
p_canon = unified_test(X, Y_alt, Z)                              # scale='abs' (canonical)
p_e2    = unified_test(X, Y_alt, Z, config=GFCMConfig(scale='e2'))  # non-robust squared scale
print('canonical (scale=abs):', round(p_canon, 4))
print('ablation  (scale=e2) :', round(p_e2, 4))
print()
print(GFCMConfig())  # every field + its canonical default

## 3. Mixed-type data

Categorical variables are handled natively (one-hot residual bank). Pass **integer-coded** columns (label-encode strings first). Types are auto-detected (integer with <= 15 distinct levels -> categorical) or declared explicitly via `x_cat` / `y_cat` / `z_cat`.

In [ ]:
Xc = rng.integers(0, 3, n).astype(float)                 # 3-level categorical X
print('auto-detected    :', round(unified_test(Xc, Y_alt, Z), 4))
print('declared x_cat   :', round(unified_test(Xc, Y_alt, Z, x_cat=True), 4))

Zmix = np.column_stack([rng.normal(size=n), rng.integers(0, 3, n).astype(float)])
print('categorical Z col:', round(unified_test(X, Y_alt, Zmix, z_cat=[False, True]), 4))

## 4. Inside constraint-based discovery (PC)

PC/FCI issue thousands of CI tests on subsets of one dataset. **Resolve the type mask once** at the dataset level with `infer_types`, then thread it -- never rely on per-call auto-detection (it re-infers per subsample). `GFCMCITest` packages exactly this as a cbcd-compatible `CITest`.

In [ ]:
from gfcm_citest import GFCMCITest

data = np.column_stack([X, Y_alt, Z])
is_cat = infer_types(data)                 # bool[p], resolved ONCE for the whole dataset
print('is_cat:', is_cat.tolist())

ci = GFCMCITest(data, cat=is_cat)          # types fixed once; PC looks them up by column index
print('test(0, 1, [2, 3]) p =', round(ci(0, 1, [2, 3]), 4))

try:
    import cbcd
    cpdag = cbcd.pc(data, ci_test=ci)      # PC threads indices; the adapter supplies typed CI tests
    print('PC ran; CPDAG:', cpdag)
except ImportError:
    print('cbcd not installed here; see experiments/gfcm_citest.py for the adapter')